In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.impute import KNNImputer
from functools import reduce
from pathlib import Path
from dotenv import load_dotenv
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from datetime import datetime
from itertools import combinations
from pandas.api.types import is_integer_dtype
from collections import Counter, defaultdict
from math import ceil

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

low_frequency_path = OUT_DIR / "handled_low_frequency_categories.csv"

# Let pandas detect the delimiter
data = pd.read_csv(low_frequency_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data



In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
# NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)
print(nan_sums)

# Missing Values Imputation Strategy

In [ ]:
# -----------------------------------------------------------------------------
# Missing Value Imputation Plan — with Verified Missingness Overview
#
# Context:
#   Dataset includes numeric, categorical, and date-like fields (Datum columns).
#
# Dataset size:
#   N = 47,946 rows
#
# Missingness Summary (by column group)
#
#   Orientation participation (numeric / date identifiers):
#     • sdo2_orientation_First_Event_Date ................. 38,720  (80.74%)
#     • sdo2_orientation_Last_Event_Date .................. 38,720  (80.74%)
#     • sdo2_orientation_Number_of_Events_Attended ........ 38,720  (80.74%)
#     • sdo2_orientation_STUDENTNUMMER .................... 38,720  (80.74%)
#     • sdo2_orientation_Number_of_Event_Types ............ 38,720  (80.74%)
#
#   Academic performance (numeric):
#     • sdo6_results_Average_Grade_B ...................... 11,542  (24.08%)
#     • sdo6_results_Average_Grade_A ......................  7,592  (15.83%)
#     • sdo6_results_Percentage_Credits_B .................  6,428  (13.41%)
#     • sdo6_results_Potential_Credits_B ..................  6,028  (12.57%)
#     • sdo6_results_Percentage_Credits_A .................  4,319  ( 9.01%)
#     • sdo6_results_Potential_Credits_A ..................  4,158  ( 8.67%)
#
#   SKC and event-related dates:
#     • sdo2_skc_SKC_DATUM ................................  5,248  (10.94%)
#
#   Distances (numeric):
#     • sdo1_prev_distance_distance_to_3584_CS ............  4,536  ( 9.46%)
#     • sdo1_prev_distance_distance_to_3812_PA ............  4,536  ( 9.46%)
#     • sdo1_postal_distance_distance_to_3584_CS ..........  1,535  ( 3.20%)
#     • sdo1_postal_distance_distance_to_3812_PA ..........  1,535  ( 3.20%)
#
#   Demographics (numeric / categorical):
#     • sdo1_characteristics_age_start_study ..............  3,476  ( 7.25%)
#     • sdo1_characteristics_Dutch_nationality ............  3,476  ( 7.25%)
#
#   Previous education (date):
#     • sdo1_previous_Final_Exam_Date .....................    779  ( 1.62%)
#
#   Degree-level and indicator columns (mostly complete):
#     • sdo5_degree_POSTAL_COUNTRY_NL .....................      5  (0.01%)
#     • All other sdo5_degree_* and “has_*” indicator fields .... 0  (0%)
#
# -----------------------------------------------------------------------------
# Imputation Plan
#
# 1) Orientation participation (≈81% missing — structural absence):
#      → Do not impute dates (keep as NaT).
#      → Numeric counts → 0.
#      → Structural missingness handled via has_orientation flag.
#
# 2) Academic performance (≈9–24% missing, NOT random):
#      → Missing values mean the student never enrolled for any exam.
#      → Therefore, impute all NaN values with 0 (not median).
#      → Add <col>_missing_flag for interpretability.
#
# 3) Distances (≈3–9% missing):
#      → Numeric distance columns → median imputation.
#      → Add <col>_missing_flag.
#
# 4) Demographics (≈7% missing):
#      → Age → median; Nationality → mode.
#      → Add missing flags.
#
# 5) Dates (partial missingness):
#      → Convert to datetime (errors='coerce').
#      → For sdo2_skc_SKC_DATUM and sdo1_previous_Final_Exam_Date:
#            - Impute with median date (preserves chronological order).
#      → For structurally missing orientation event dates:
#            - Keep NaT (represents “no event recorded”).
#
# 6) Columns with no missing values:
#      → No action required.
#
# -----------------------------------------------------------------------------
# Summary:
#   - Structural missingness (no participation or exams) retained as NaT or 0.
#   - Academic performance NaN values → 0 (reflects true non-enrollment).
#   - Random numeric gaps filled with medians; categorical with modes.
#   - Partial date gaps imputed with median date for temporal continuity.
#   - All imputations preserve feature meaning, interpretability, and integrity.
# -----------------------------------------------------------------------------


In [ ]:
# =============================================================================
# PURPOSE
# -----------------------------------------------------------------------------
# Function: impute_all(data)
#
# Structured missing-value imputation for the student dataset (N = 47,946),
# preserving structural missingness (e.g., no orientation activity, no exams).
#
# Key imputations (per verified plan):
#   1) Orientation (~81% missing, structural):
#        - Numeric counts → fill with 0.
#        - Dates → keep as NaT (no imputation).
#   2) Academic performance (non-random missingness = no exams):
#        - All NaN → fill with 0 (NOT median).
#   3) Distances (≈3–9% missing):
#        - Numeric → median.
#   4) Demographics (≈7% missing):
#        - Age → median; Dutch nationality → mode.
#   5) Dates with partial missingness:
#        - sdo2_skc_SKC_DATUM, sdo1_previous_Final_Exam_Date → median date.
#
# Flags:
#   For every column imputed or kept as structural NaT, add:
#     <column>_missing_flag  (1 if originally missing, else 0)
#
# New flag columns added: 19
#   - Orientation: 5 (3 counts + 2 dates)
#   - Academic performance: 6
#   - Distances: 4
#   - Demographics: 2
#   - Other dates (SKC, Final Exam): 2
#
# Output:
#   Returns a new DataFrame with imputations applied and *_missing_flag columns.
# =============================================================================

def impute_all(data: pd.DataFrame) -> pd.DataFrame:
    df = data.copy()

    # -----------------------------
    # 1) Define column groups
    # -----------------------------
    # Orientation: numeric counts (set to 0 if missing)
    orientation_count_cols = [
        "sdo2_orientation_Number_of_Event_Types",
        "sdo2_orientation_Number_of_Events_Attended",
        "sdo2_orientation_STUDENTNUMMER",
    ]
    # Orientation: dates (structural NaT)
    orientation_date_cols_structural = [
        "sdo2_orientation_First_Event_Date",
        "sdo2_orientation_Last_Event_Date",
    ]

    # Dates with partial missingness → impute median date
    date_cols_median = [
        "sdo2_skc_SKC_DATUM",
        "sdo1_previous_Final_Exam_Date",
    ]

    # Academic performance → fill with 0 (non-random missingness)
    academic_numeric = [
        "sdo6_results_Average_Grade_B",
        "sdo6_results_Average_Grade_A",
        "sdo6_results_Percentage_Credits_B",
        "sdo6_results_Potential_Credits_B",
        "sdo6_results_Percentage_Credits_A",
        "sdo6_results_Potential_Credits_A",
    ]

    # Distances → median (drop removed *_postcode columns)
    distance_numeric = [
        "sdo1_prev_distance_distance_to_3584_CS",
        "sdo1_prev_distance_distance_to_3812_PA",
        "sdo1_postal_distance_distance_to_3812_PA",
        "sdo1_postal_distance_distance_to_3584_CS",
    ]

    # Demographics
    demo_age_median = ["sdo1_characteristics_age_start_study"]
    demo_nat_mode   = ["sdo1_characteristics_Dutch_nationality"]

    all_date_cols = list(dict.fromkeys(orientation_date_cols_structural + date_cols_median))

    # -----------------------------
    # 2) Helpers
    # -----------------------------
    def add_flag(col: str):
        flag = f"{col}_missing_flag"
        df[flag] = df[col].isna().astype("int8")

    def to_datetime_cols(cols):
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_datetime(df[c], errors="coerce")

    def median_datetime(series: pd.Series) -> pd.Timestamp:
        # pandas supports .median() on datetime64[ns]
        return series.median()

    def mode_safe(series: pd.Series):
        m = series.mode(dropna=True)
        return m.iloc[0] if len(m) > 0 else np.nan

    def coerce_numeric(cols):
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce")

    # -----------------------------
    # 3) Convert datatypes
    # -----------------------------
    to_datetime_cols(all_date_cols)
    # NOTE: do NOT coerce nationality to numeric; we will impute by mode
    coerce_numeric(orientation_count_cols + academic_numeric + distance_numeric + demo_age_median)

    # -----------------------------
    # 4) Orientation (structural)
    # -----------------------------
    for c in orientation_count_cols:
        if c in df.columns:
            add_flag(c)
            df[c] = df[c].fillna(0)

    for c in orientation_date_cols_structural:
        if c in df.columns:
            add_flag(c)  # keep NaT (no imputation)

    # -----------------------------
    # 5) Dates (partial missingness → median date)
    # -----------------------------
    for c in date_cols_median:
        if c in df.columns:
            add_flag(c)
            med = median_datetime(df[c])
            if not pd.isna(med):
                df[c] = df[c].fillna(med)

    # -----------------------------
    # 6) Academic performance → ZERO (not median)
    # -----------------------------
    for c in academic_numeric:
        if c in df.columns:
            add_flag(c)
            df[c] = df[c].fillna(0)

    # -----------------------------
    # 7) Distances → median
    # -----------------------------
    for c in distance_numeric:
        if c in df.columns:
            add_flag(c)
            med = df[c].median(skipna=True)
            df[c] = df[c].fillna(med)

    # -----------------------------
    # 8) Demographics
    # -----------------------------
    # Age → median
    for c in demo_age_median:
        if c in df.columns:
            add_flag(c)
            med = df[c].median(skipna=True)
            df[c] = df[c].fillna(med)

    # Dutch nationality → mode
    for c in demo_nat_mode:
        if c in df.columns:
            add_flag(c)
            m = mode_safe(df[c])
            df[c] = df[c].fillna(m)

    # -----------------------------
    # 9) Clean-up for numeric consistency
    # -----------------------------
    if "sdo2_orientation_STUDENTNUMMER" in df.columns:
        # Use pandas nullable integer to be safe even if future NaNs appear
        df["sdo2_orientation_STUDENTNUMMER"] = df["sdo2_orientation_STUDENTNUMMER"].astype("Int64")

    return df

# Example usage:
data = impute_all(data)


In [ ]:
# Confirm NaN count per column in data
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False).head(20)
print(nan_sums)

In [ ]:
#del data ['sdo2_orientation_Last_Event_Date']
#del data ['sdo2_orientation_Last_Event_Date']

In [ ]:
data.columns

In [ ]:
data.shape

In [ ]:
print(data['sdo5_degree_POSTAL_COUNTRY_NL'].replace('', np.nan).dropna().value_counts())

In [ ]:
data.to_csv(f"{OUT_DIR}/missing_values_imputed.csv", index=False)
print(f"✅ Saved to: {OUT_DIR}/missing_values_imputed.csv")